# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [70]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-02-22T19:58:32.153135",
    "last_interaction": "2026-02-22T19:58:32.153180",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-02-22T19:58:32.025279",
    "last_interaction": "2026-02-22T19:58:32.025614",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:cb3cc99e-19a3-4f12-897d-5d6f4ed670ca",
    "dctIssued": "2026-02-22T19:58:32.194927Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [71]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:08f0

## Provider creates initial offer (Provider -> Consumer)

In [72]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba"
    },
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:cd42efeb-7510-4214-91

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [73]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba"
    },
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:08f0d5e6-28f3-4da4-9f90-6dc2178ceb8e",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [74]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba"
    },
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:cd42efeb-7510-4214-9109-f660ed9f86fe",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [75]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:08f0d5e6-28f3-4da4-9f90-6dc2178ceb8e",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:59:24.415376Z",
    "updatedAt": "2026-02-22T20:59:25.295841Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:4

## Provider creates the Agreement (Provider -> Consumer)

In [76]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:cd42efeb-7510-4214-9109-f660ed9f86fe",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:59:24.390021Z",
    "updatedAt": "2026-02-22T20:59:25.556741Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:74e9b

## Consumer verifies the agreement (Consumer -> Provider)

In [77]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:08f0d5e6-28f3-4da4-9f90-6dc2178ceb8e",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:59:24.415376Z",
    "updatedAt": "2026-02-22T20:59:25.611566Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:4

## Provider finalizes the negotiation (Provider -> Consumer)

In [78]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:4a54495d-ca69-40c0-af90-af29750d8043",
    "providerPid": "urn:provider-pid:74e9b983-9fed-424a-8d4d-37dca397b4f0",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:cd42efeb-7510-4214-9109-f660ed9f86fe",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:59:24.390021Z",
    "updatedAt": "2026-02-22T20:59:25.663039Z",
    "identifiers": {
      "providerPid": "urn:provider-pid

## Final agreement

In [79]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
  "negotiationAgentProcessId": "urn:negotiation-process:cd42efeb-7510-4214-9109-f660ed9f86fe",
  "negotiationAgentMessageId": "urn:negotiation-message:ec96f5c9-56a2-426d-be29-3022e0662f1e",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba",
    "timestamp": "1771793965"
  },
  "target": "urn:dataset:5c4c97ea-37c9-47cb-9cce-ba17105798ba",
  "state": "ACTIVE",
  "createdAt": "2026-02-22T20:59:25.565505Z",
  "updatedAt": "2026-02-22T20:59:25.668509Z"
}

Final agreement id: 
urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [80]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:2d5981d7-1958-4354-bc1a-707321bbe5ac",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "callbackAddress": "http://127.0.0.1:

## Start transfer (Provider -> Consumer)

In [81]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:d9f6f058-bb13-4739-854c-da0576dc3b5f",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:db2e8f8b-23c1-4986-8664-aa71a801f989",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1a

## Suspend transfer (Consumer -> Provider)

In [82]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:2d5981d7-1958-4354-bc1a-707321bbe5ac",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
   

## Restart transfer (Consumer -> Provider)

In [83]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1200/dataplane/proxy/urn:dataplane-transfer:f84407b6-0d3d-4cb1-b8c0-0285854193fb",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:2d5981d7-1958-4354-bc1a-707321bbe5ac",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [84]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:db2e8f8b-23c1-4986-8664-aa71a801f989",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
   

## Failure Test: Attempt start with invalid parameters

In [85]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2"
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "HTTP Error 400 Bad Request: {\"@context\":[\"https://w3id.org/dspace/2025/1/context.jsonld\"],\"@type\":\"TransferError\",\"consumerPid\":null,\"providerPid\":null,\"code\":\"6030\",\"reason\":[\"TransferProcessMessageType TransferStartMessage is not allowed here. Current state is SUSPENDED ByProvider\",\"Failed to parse file\"]}"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [86]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "6030",
    "reason": [
      "TransferProcessMessageType TransferSuspensionMessage is not allowed here. Current state is SUSPENDED",
      "Failed to parse file"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [87]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:dc72ceda-7f9f-4602-b49f-2fee563dea75",
    "providerPid": "urn:provider-pid:3b5c3e8e-0e48-4516-a003-ca622d0d35f2",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:db2e8f8b-23c1-4986-8664-aa71a801f989",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:2a302c1a-75c9-48ee-916f-b4c2c22cb8c9",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:59:28.705764Z",